# Agent Evaluation

_Assessing agents performance_

Uses an built-in evaluator (that takes agent outputs and optionally reference outputs, and returns a score) to evaluate an agent by measuring how well the agent performs by assessing its execution trajectory, the sequence of messages and tool calls it produces using approaches such as _Trajectory Match_ (deterministic comparison) and _LLM-as-judge_ (qualitative assessment).


**Prerequisites:**

Follow the instruction below.

1. Open a new **Anaconda Prompt** terminal and activate environment `agentic_ai` using command `conda activate agentic_ai`.

2. Refer to the package installtion section in GitHub README.md and install LangChain `agentevals` in the same environment.

In [1]:
# Imports packages

from typing import Literal
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from agentevals.trajectory import create_trajectory_match_evaluator
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, \
    TRAJECTORY_ACCURACY_PROMPT, TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE

## Model

_Connecting an appropriate model to a chat client._

In [ ]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_MODEL_AS_JUDGE = "granite4.1:3b"     # "qwen3.5:2b" found causing error in generating data in valid JSON
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [ ]:
# Initializes chat client for agent
model = ChatOllama(
    model=OLLAMA_MODEL,
    reasoning=False,
    base_url=OLLAMA_ENDPOINT)

# Initializing model client to be used as judge during evaluation
model_as_judge = ChatOllama(
    model=OLLAMA_MODEL_AS_JUDGE,
    reasoning=False,
    base_url=OLLAMA_ENDPOINT)

## Tools

_Tools the agent needs to complete its tasks._

For this experiment, all tools are dummy. But actual function implementation would be required for production use cases.

In [ ]:
@tool
def get_weather(city: str):
    """
    Get weather information for a city.
    
    Parameters:
    - city (str): The name of the city to get weather information for.
    """
    
    return f"It's 75 degrees and sunny in {city}."


@tool
def get_events(city: str):
    """
    Get events happening in a city.
    
    Parameters:
    - city (str): The name of the city to get events for.
    """
    
    return f"Concert at the park in {city} tonight."


@tool
def get_detailed_forecast(city: str):
    """
    Get detailed weather forecast for a city.
    
    Parameters:
    - city (str): The name of the city to get detailed forecast for.
    """
    
    return f"Detailed forecast for {city}: sunny all week."

In [ ]:
# Creates agent to complete its task

agent = create_agent(model, tools=[get_weather, get_events, get_detailed_forecast])

## Evaluation using Trajectory Match

_Judging the trajectory of an agent's execution against an expected trajectory._

In [ ]:
def test_trajectory_match(
    query: str,
    reference_trajectory: list,
    trajectory_match_mode=Literal["strict", "unordered", "subset", "superset"]):
    """
    Helper function to deterministically test trajectory match against reference trajectory.

    Parameters:
    - query (str): The input query to the agent.
    - reference_trajectory (list): The expected trajectory of messages.
    - trajectory_match_mode (str): The mode of trajectory matching. Options are "strict", "unordered", "subset", "superset".

    Returns:
    - dict: A dictionary containing the evaluation results.
    """

    result = agent.invoke({"messages": [HumanMessage(content=query)]})
    
    evaluator = create_trajectory_match_evaluator(trajectory_match_mode=trajectory_match_mode)

    return evaluator(outputs=result["messages"], reference_outputs=reference_trajectory)

**Strict mode match**

_Matching exact message structure and tool calls in the same order (message content can differ)._

In [ ]:
query = "What's the weather in San Francisco?"      # Sets end-user query

# Sets reference trajectory the agent's trajectory would be compared with
reference_trajectory = [
        HumanMessage(content="What's the weather in San Francisco?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}}
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        AIMessage(content="The weather in San Francisco is 75 degrees and sunny."),
    ]

# Performs agent's trajectory match
eval_result = test_trajectory_match(query, reference_trajectory, "strict")

print(eval_result)      # Prints the evaluation

{'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


Refer to the value associated with key 'score' where it indicates 100% or 0% match against value 'True' or 'False', respectively, as part of the agent trajectory evaluation.

In [ ]:
# To analyze the score the evaluation provides by manually checking the trajectory
# by invoking the agent for the task
agent.invoke({"messages": [HumanMessage(content=query)]})

{'messages': [HumanMessage(content="What's the weather in San Francisco?", additional_kwargs={}, response_metadata={}, id='252a31b4-bf3f-49b1-b780-7763a844df77'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-30T10:07:27.918193507Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8747749215, 'load_duration': 202230783, 'prompt_eval_count': 278, 'prompt_eval_duration': 7162986634, 'eval_count': 18, 'eval_duration': 1358908464, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f17ff-5c7f-73b3-8593-d06759c71e63-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '7892145a-f928-479f-99d6-4ccedf007c9e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 18, 'total_tokens': 296}),
  ToolMessage(content="It's 75 degrees and sunny in San Francisco.", name='get_weather', id='e31d15b8-21ff-403c-a771-6fa003

**Unordered mode match**

_Same as "strict" mode match, but allows tool calls in any order._

In [ ]:
query = "What's happening in San Francisco today? What's the weather there?"    # Sets end-user query

# Sets reference trajectory the agent's trajectory would be compared with
reference_trajectory = [
        HumanMessage(content="What's happening in San Francisco today? What's the weather there?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}},
            {"id": "call_2", "name": "get_events", "args": {"city": "San Francisco"}},
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        ToolMessage(content="Concert at the park in San Francisco tonight.", tool_call_id="call_2"),
        AIMessage(content="Today in San Francisco: 75 degrees and sunny with a concert at the park tonight."),
    ]

# Performs agent's trajectory match
eval_result = test_trajectory_match(query, reference_trajectory, "unordered")

print(eval_result)      # Prints the evaluation

{'key': 'trajectory_unordered_match', 'score': True, 'comment': None, 'metadata': None}


In [20]:
agent.invoke({"messages": [HumanMessage(content=query)]})

{'messages': [HumanMessage(content="What's happening in SF today? What's the weather there?", additional_kwargs={}, response_metadata={}, id='b4d45861-8be2-4b75-ad93-3fafa81597c8'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-30T10:32:25.186916132Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11011510394, 'load_duration': 174550898, 'prompt_eval_count': 283, 'prompt_eval_duration': 8029807516, 'eval_count': 36, 'eval_duration': 2764451969, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f1816-2c5d-7260-bef4-d0c68733e3a1-0', tool_calls=[{'name': 'get_events', 'args': {'city': 'San Francisco'}, 'id': 'd5bafc49-ce08-4f52-843c-c6582bf1b631', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '39c7be4e-e15b-4dff-9f7f-3f0eac94220b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 283, 'output_tokens': 36, 

**Exercises: Subset & Superset mode match**

Try on your own to test partial trajectory matching using subset and superset mode.
The subset mode ensures the agent did not call any tools beyond those in the reference.
The superset mode verifies that the agent called at least the tools in the reference trajectory, allowing additional tool calls. 

In [ ]:
# Code here

## Evaluating using LLM-as-Judge
_Using LLM as a judge to evaluate the trajectory (where reference trajectory is optional unlike the trajectory match evaluation)._


In [ ]:
def test_trajectory_quality(
    query: str,
    reference_trajectory: list = None):
    """
    Helper function to assess overall quality and reasoning of the trajectory without strict expectations using an LLM as a judge.

    Parameters:
    - query (str): The input query to the agent.
    - reference_trajectory (list): The expected trajectory of messages. If provided, the judge will compare the agent's trajectory against this reference.

    Returns:
    - dict: A dictionary containing the evaluation results.
    """

    result = agent.invoke({"messages": [HumanMessage(content=query)]})
    
    evaluator = create_trajectory_llm_as_judge(
        judge=model_as_judge,        
        prompt= TRAJECTORY_ACCURACY_PROMPT if reference_trajectory is None else TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
        )

    return evaluator(outputs=result["messages"])

In [7]:
query = "What's the weather in Seattle?"

eval_result = test_trajectory_quality(query)

print(eval_result)

{'key': 'trajectory_accuracy', 'score': True, 'comment': "The trajectory logically progresses from the user’s initial query about the weather in Seattle to calling the get_weather tool with the appropriate argument (city: 'Seattle'). The final response correctly provides the current weather conditions, making it clear and relevant. All steps are coherent and lead directly to achieving the goal of informing the user about Seattle's weather.", 'metadata': None}


Refer to the values associated with both the keys 'score' and 'comment'. The 'comment' contains the explanation behind the score.